# Chapter 6: Fine-Tuning for Classification

Adapting a pretrained GPT model into a text classifier (spam detection).

**What we'll do:**
1. Create a small spam/ham classification dataset
2. Build a GPT-based classifier with a classification head
3. Compare random init vs pretrained backbone
4. Fine-tune and watch accuracy improve
5. Classify real messages

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import torch.nn.functional as F
import tiktoken

torch.manual_seed(42)
device = 'mps' if torch.backends.mps.is_available() else \
         'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. The Dataset

We use a synthetic spam detection dataset — 50 examples (25 spam, 25 ham).
Small, but enough to demonstrate the concept.

In [ ]:
from ch06.dataset import SPAM_DATASET, create_classification_loaders

# Quick look at the data
print(f'Total samples: {len(SPAM_DATASET)}')
print(f'Spam: {sum(1 for _, l in SPAM_DATASET if l == 1)}')
print(f'Ham:  {sum(1 for _, l in SPAM_DATASET if l == 0)}')
print()
print('Sample spam:', SPAM_DATASET[0][0])
print('Sample ham: ', SPAM_DATASET[-1][0])

In [ ]:
# Create train/val/test splits
train_loader, val_loader, test_loader = create_classification_loaders(
    max_len=64, batch_size=8
)

# Inspect a batch
for token_ids, labels in train_loader:
    print(f'Token IDs shape: {token_ids.shape}')  # (8, 64)
    print(f'Labels:          {labels.tolist()}')
    break

## 2. The Classifier Architecture

We take the GPT backbone from ch04 and replace the language model head with a classification head:

```
GPT Backbone → Hidden States → Last Token → Linear(d_model → 2) → Spam/Ham
```

In [ ]:
from ch05.pretrain import SMALL_CONFIG
from ch06.classifier import GPTClassifier

print(f'Model config: d_model={SMALL_CONFIG.d_model}, '
      f'n_heads={SMALL_CONFIG.n_heads}, n_layers={SMALL_CONFIG.n_layers}')

# Compare freeze strategies
for n in [0, 1, 2, -1]:
    clf = GPTClassifier(SMALL_CONFIG, num_classes=2, unfreeze_last_n=n)
    label = 'all' if n == -1 else str(n)
    total = clf.count_total_parameters()
    trainable = clf.count_trainable_parameters()
    print(f'  Unfreeze {label:>3s} blocks: {trainable:>8,} / {total:,} trainable ({trainable/total:.1%})')

## 3. Before Fine-Tuning (Baseline)

With random weights, the model should perform at chance level (~50% for 2 classes).

In [ ]:
from ch06.train_classifier import calc_accuracy, classify_text

# Random init model
model = GPTClassifier(SMALL_CONFIG, num_classes=2, unfreeze_last_n=-1)

# Try loading pretrained backbone
checkpoint_path = os.path.join('..', 'ch05', 'checkpoints', 'final.pt')
if os.path.exists(checkpoint_path):
    model.load_pretrained_backbone(checkpoint_path, device=device)
    print('Loaded pretrained backbone!')
else:
    print('No pretrained checkpoint — using random init')

model.to(device)
before_acc = calc_accuracy(model, test_loader, device)
print(f'\nTest accuracy BEFORE fine-tuning: {before_acc:.1%}')
print(f'(Random chance: 50%)')

In [ ]:
# Try classifying some messages before training
test_msgs = [
    'You won a free iPhone! Click here!',
    'Hey, want to grab coffee later?',
]
labels = {0: 'ham', 1: 'spam'}
print('Before fine-tuning:')
for msg in test_msgs:
    pred, probs = classify_text(model, msg, device=device)
    print(f'  [{labels[pred]}] ({max(probs)*100:.0f}%) {msg}')

## 4. Fine-Tuning

Now let's train the classifier. Key settings:
- **Low learning rate** (5e-5) — don't destroy pretrained features
- **Only head + last block trainable** — efficient transfer learning
- **20 epochs** — small dataset, converges quickly

In [ ]:
from ch06.train_classifier import train_classifier

history = train_classifier(
    model, train_loader, val_loader,
    num_epochs=30,
    lr=5e-4,
    device=device,
)

## 5. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
epochs = range(1, len(history['train_losses']) + 1)
ax1.plot(epochs, history['train_losses'], 'b-', label='Train')
ax1.plot(epochs, history['val_losses'], 'r-', label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Classification Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs, history['train_accs'], 'b-', label='Train')
ax2.plot(epochs, history['val_accs'], 'r-', label='Val')
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Classification Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. After Fine-Tuning

Let's see how the model performs now!

In [ ]:
after_acc = calc_accuracy(model, test_loader, device)
print(f'Test accuracy AFTER fine-tuning: {after_acc:.1%}')
print(f'Improvement: {after_acc - before_acc:+.1%}')

In [ ]:
# Classify messages after training
test_messages = [
    'Congratulations! You won a free cruise. Click to claim!',
    'Hey, want to grab lunch tomorrow?',
    'URGENT: Verify your account or it will be suspended!',
    'The meeting has been rescheduled to Thursday.',
    'Make $10000 a day with this simple trick!',
    'Thanks for the birthday wishes!',
    'Your package has been shipped and will arrive Tuesday.',
    'LIMITED TIME: 90% off everything! Act now!!!',
]

label_names = {0: 'ham  ✉️', 1: 'spam 🚫'}
print('Live Classification Demo:\n')
for msg in test_messages:
    pred, probs = classify_text(model, msg, device=device)
    confidence = max(probs) * 100
    print(f'  [{label_names[pred]}] ({confidence:.0f}%) {msg}')

## 7. Before vs After Comparison

| Metric | Before | After |
|--------|--------|-------|
| Test Accuracy | ~50% (random) | >>50% |
| Can distinguish spam? | No | Yes |
| Trainable params | N/A | Only head + last block |

### Key Takeaways

1. **Transfer learning works** — even our tiny GPT model learns to classify with few examples
2. **Freezing helps** — we only trained ~20% of parameters but got good results
3. **The last token trick** — causal attention makes the last position ideal for classification
4. **Low LR is crucial** — 5e-5 (not 5e-4) to preserve pretrained knowledge